# Exercise 1: A frozen baseline

**Level: introductory worked exercise.** The “Your turn” task is an extension.

*Speech Technologies · version 0.1.0*

> Measure the unadapted Whisper baseline on a frozen Kazakh test set: raw and normalised WER and CER, total duration, results per speaker and error examples. With the Hands-On script, compare it with the adapted checkpoint on the same items, choosing all decoding settings on validation.

**Setup.** `pip install sherpa-onnx==1.13.8 librosa==0.11.0 soundfile requests numpy`. Everything runs on the CPU without a GPU or the Hugging Face Hub:

- **ASR:** OpenAI Whisper *small*, as the int8 ONNX export published in the sherpa-onnx GitHub release `asr-models` (639 MB archive), decoded greedily by sherpa-onnx.
- **TTS:** the Piper VITS voice `kk_KZ-issai-high` (six speakers, trained from scratch on the ISSAI KazakhTTS data, CC BY 4.0 according to its model card) and, in Exercise 4, `tr_TR-dfki-medium` (one speaker, CC BY-NC-SA 4.0), from the sherpa-onnx release `tts-models`.
- **Text:** the Kazakh and Turkish sides of the X-WMT English–Kazakh and English–Turkish test sets in the commit-pinned TIL release (MIT licence), with the spaces around punctuation removed.

Archives are checked against the SHA-256 digests in the code and cached in `BOOK_DATA_DIR` (default `~/.cache/turkic-nlp-book`), together with every synthesised waveform and every transcript, so a rerun reuses them. **The test speech is synthetic.** It comes from a TTS model trained on read studio speech; its error rates say nothing about Whisper on real Kazakh recordings, and they are used here only to compare conditions on identical input. Piper samples its output (noise scale 0.667), so a fresh cache produces slightly different audio and slightly different numbers.

The test set is fixed before anything is decoded: the first 60 X-WMT Kazakh sentences with 5–20 words and no digits, each synthesised once by one of the six Piper voices in turn. Whisper is told the language (`kk`) and decodes greedily. Scoring uses the functions of `code/remaining/speech.py`, so the numbers are computed exactly as the Hands-On script computes them.

In [ ]:
import collections, hashlib, json, os, re, statistics, tarfile, time, unicodedata
from pathlib import Path
import numpy as np, requests

TIL = "https://raw.githubusercontent.com/turkic-interlingua/til-mt/7d988cc47dba889d02262d893bf9a46e43a6bb36/"
SHERPA = "https://github.com/k2-fsa/sherpa-onnx/releases/download/"
ARCHIVES = {   # sherpa-onnx release archive -> SHA-256 of the file used for the book's run
    "asr-models/sherpa-onnx-whisper-small.tar.bz2": "486a46afbb7ba798507190ffe02fea2dd726049af212e774537efac6afb210a6",
    "tts-models/vits-piper-kk_KZ-issai-high.tar.bz2": "6c59955f7f5e3fc50b130ced59a430eb4ef62d69155af4ce836db0dde4a42682",
    "tts-models/vits-piper-tr_TR-dfki-medium.tar.bz2": "b63babaefcdd63202c7079aba25b72a2013825ad5895e2aeff03d1ae5a5b3bbf",
}
CACHE = Path(os.environ.get("BOOK_DATA_DIR", "~/.cache/turkic-nlp-book")).expanduser()
FETCHED = {}                                   # url -> sha256 of every file read (saved with the outputs)
SR = 16000                                     # Whisper's sampling rate
THREADS = os.cpu_count() or 1

def fetch(url):
    path = CACHE / hashlib.sha1(url.encode()).hexdigest()[:12] / url.rsplit("/", 1)[-1]
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        with requests.get(url, timeout=600, stream=True) as r:
            r.raise_for_status()
            with open(str(path) + ".part", "wb") as f:
                for block in r.iter_content(1 << 20):
                    f.write(block)
        os.replace(str(path) + ".part", path)
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    FETCHED[url] = h.hexdigest()
    return path

def archive(name):
    """Download, verify and unpack a sherpa-onnx release archive; return the unpacked folder."""
    path = fetch(SHERPA + name)
    if FETCHED[SHERPA + name] != ARCHIVES[name]:
        raise ValueError(f"{name}: the release file has changed; check it before using it")
    root = path.parent / path.name.removesuffix(".tar.bz2")
    if not root.exists():
        with tarfile.open(path) as t:
            t.extractall(path.parent, filter="data")
    return root

def detok(s):
    """Undo the X-WMT tokenisation (spaces around punctuation)."""
    s = re.sub(r" ([’'] ?)", lambda m: m.group(1).strip(), s)
    s = re.sub(r" ([.,!?;:%)\]»])", r"\1", s)
    return re.sub(r"([(\[«]) ", r"\1", s).strip()

def xwmt_side(pair, lang):
    """One side of an X-WMT test set, detokenised."""
    text = fetch(f"{TIL}xwmt/test/{pair}/{pair}.{lang}").read_text(encoding="utf-8").splitlines()
    return [detok(s) for s in text if s.strip()]

def test_sentences(pair, lang, n):
    """The first n sentences with 5-20 words and no digits (number expansion is a separate problem)."""
    return [s for s in xwmt_side(pair, lang) if not re.search(r"\d", s) and 5 <= len(s.split()) <= 20][:n]

# ---- TTS: Piper VITS voices through sherpa-onnx; every waveform is cached on disk -------------
VOICES = {"kk": "kk_KZ-issai-high", "tr": "tr_TR-dfki-medium"}
_TTS = {}

def tts(voice, noise=None):
    if (voice, noise) not in _TTS:
        import sherpa_onnx
        root = archive(f"tts-models/vits-piper-{voice}.tar.bz2")
        vits = sherpa_onnx.OfflineTtsVitsModelConfig(model=str(root / f"{voice}.onnx"), tokens=str(root / "tokens.txt"),
                                                     data_dir=str(root / "espeak-ng-data"))
        if noise is not None:                   # noise=0 makes the voice deterministic
            vits.noise_scale = vits.noise_scale_w = noise
        _TTS[voice, noise] = sherpa_onnx.OfflineTts(sherpa_onnx.OfflineTtsConfig(
            model=sherpa_onnx.OfflineTtsModelConfig(vits=vits, num_threads=THREADS)))
    return _TTS[voice, noise]

def speech(text, sid=0, lang="kk", draw=0, noise=None):
    """16 kHz float32 synthetic speech for text; `draw` indexes repeated samples of the same input."""
    key = hashlib.sha1(json.dumps([VOICES[lang], sid, draw, noise, text], ensure_ascii=False).encode()).hexdigest()
    path = CACHE / "ch11-audio" / f"{key}.npy"
    if not path.exists():
        import librosa
        a = tts(VOICES[lang], noise).generate(text, sid=sid, speed=1.0)
        w = librosa.resample(np.asarray(a.samples, dtype=np.float32), orig_sr=a.sample_rate, target_sr=SR)
        path.parent.mkdir(parents=True, exist_ok=True)
        np.save(path, w.astype(np.float32))
    return np.load(path)

# ---- ASR: Whisper small (int8 ONNX), greedy; every transcript is cached ----------------------
_ASR = {}
ASR_CACHE = CACHE / "ch11-transcripts.jsonl"

def _asr(lang):
    if lang not in _ASR:
        import sherpa_onnx
        root = archive("asr-models/sherpa-onnx-whisper-small.tar.bz2")
        _ASR[lang] = sherpa_onnx.OfflineRecognizer.from_whisper(
            encoder=str(root / "small-encoder.int8.onnx"), decoder=str(root / "small-decoder.int8.onnx"),
            tokens=str(root / "small-tokens.txt"), language=lang, task="transcribe", num_threads=THREADS)
    return _ASR[lang]

def _transcripts():
    out = {}
    if ASR_CACHE.exists():
        for line in ASR_CACHE.read_text(encoding="utf-8").splitlines():
            r = json.loads(line); out[r["key"]] = r
    return out
TRANSCRIPTS = _transcripts()

def transcribe(wave, lang="kk"):
    """(text, language Whisper used, decoding seconds); lang="" lets Whisper detect the language."""
    wave = np.ascontiguousarray(wave, dtype=np.float32)
    key = hashlib.sha1(wave.tobytes()).hexdigest() + "|" + lang
    if key not in TRANSCRIPTS:
        rec = _asr(lang)
        s = rec.create_stream(); s.accept_waveform(SR, wave)
        t0 = time.time(); rec.decode_stream(s)
        r = {"key": key, "text": s.result.text.strip(), "lang": s.result.lang, "sec": time.time() - t0}
        TRANSCRIPTS[key] = r
        ASR_CACHE.parent.mkdir(parents=True, exist_ok=True)
        with ASR_CACHE.open("a", encoding="utf-8") as f:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    r = TRANSCRIPTS[key]
    return r["text"], r["lang"], r["sec"]

# ---- Scoring: the same functions as code/remaining/speech.py -----------------------------------
def normalize(text):
    # Conservative scoring: preserve letters, diacritics and apostrophes.
    text = unicodedata.normalize("NFC", text).lower()
    return " ".join("".join(c if not unicodedata.category(c).startswith("P") or c in "'ʻʼ’" else " " for c in text).split())

def distance(a, b):
    row = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        nxt = [i]
        for j, y in enumerate(b, 1):
            nxt.append(min(nxt[-1] + 1, row[j] + 1, row[j - 1] + (x != y)))
        row = nxt
    return row[-1]

def errors(refs, hyps):
    """Corpus-level raw and normalised WER and CER (edits / reference words or characters)."""
    if len(refs) != len(hyps) or not refs:
        raise ValueError("unaligned/empty references")
    result = {}
    for name, fn in [("raw", lambda x: x), ("normalized", normalize)]:
        pairs = [(fn(r), fn(h)) for r, h in zip(refs, hyps)]
        for unit, split in [("wer", str.split), ("cer", list)]:
            denom = sum(len(split(r)) for r, h in pairs)
            edits = sum(distance(split(r), split(h)) for r, h in pairs)
            result[f"{name}_{unit}"] = edits / denom if denom else None
        result[f"{name}_empty_references"] = sum(not r for r, h in pairs)
    return result

def pct(x):
    return f"{100 * x:5.1f}%"

def audio_digest(waves):
    h = hashlib.sha256()
    for w in waves:
        h.update(np.ascontiguousarray(w, dtype=np.float32).tobytes())
    return h.hexdigest()[:16]

print("archives verified:", ", ".join(n.rsplit("/", 1)[-1] for n in ARCHIVES if archive(n)))

In [ ]:
kk_all = xwmt_side("en-kk", "kk")
TEST = test_sentences("en-kk", "kk", 60)               # 60 sentences, spoken by the six voices in turn
waves = [speech(s, sid=i % 6) for i, s in enumerate(TEST)]
hyps, langs, secs = zip(*(transcribe(w, "kk") for w in waves))
audio = sum(len(w) for w in waves) / SR
print(f"X-WMT en-kk test: {len(kk_all)} Kazakh sentences; the first {len(TEST)} with 5-20 words and no digits are used")
print(f"synthetic audio: {audio / 60:.1f} min, {min(len(w) for w in waves) / SR:.1f}-{max(len(w) for w in waves) / SR:.1f} s per item; audio digest {audio_digest(waves)}")
print(f"decoding: {sum(secs):.0f} s on {THREADS} CPU threads, real-time factor {sum(secs) / audio:.2f}")
r = errors(TEST, list(hyps))
print(f"Whisper small, language token kk, greedy:  raw WER {pct(r['raw_wer'])}  CER {pct(r['raw_cer'])}   "
      f"normalised WER {pct(r['normalized_wer'])}  CER {pct(r['normalized_cer'])}")
short = sum(len(normalize(h)) < 0.5 * len(normalize(r)) for r, h in zip(TEST, hyps))
print(f"hypotheses shorter than half their reference (normalised characters): {short} of {len(TEST)}")
print("normalised CER by voice:")
for sid in range(6):
    idx = [i for i in range(len(TEST)) if i % 6 == sid]
    e = errors([TEST[i] for i in idx], [hyps[i] for i in idx])
    print(f"  speaker {sid}: CER {pct(e['normalized_cer'])}  WER {pct(e['normalized_wer'])}  ({len(idx)} items)")
item = sorted(range(len(TEST)), key=lambda i: distance(list(normalize(TEST[i])), list(normalize(hyps[i]))) / len(normalize(TEST[i])))
for name, i in [("best", item[0]), ("median", item[len(item) // 2]), ("worst", item[-1])]:
    c = distance(list(normalize(TEST[i])), list(normalize(hyps[i]))) / len(normalize(TEST[i]))
    print(f"{name} item (CER {pct(c)}):\n  REF {TEST[i]}\n  HYP {hyps[i]}")

## Worked solution

Output of `experiments/ch11_exercises/run_ch11_exercises.py`:

```
X-WMT en-kk test: 700 Kazakh sentences; the first 60 with 5-20 words and no digits are used
synthetic audio: 7.0 min, 1.8-10.6 s per item; audio digest 421f2be7f3ba781d
decoding: 289 s on 2 CPU threads, real-time factor 0.69
Whisper small, language token kk, greedy:  raw WER  96.2%  CER  58.3%   normalised WER  93.4%  CER  56.4%
hypotheses shorter than half their reference (normalised characters): 19 of 60
normalised CER by voice:
  speaker 0: CER  57.5%  WER  92.6%  (10 items)
  speaker 1: CER  51.2%  WER  90.4%  (10 items)
  speaker 2: CER  61.6%  WER  97.1%  (10 items)
  speaker 3: CER  55.3%  WER  92.5%  (10 items)
  speaker 4: CER  56.6%  WER  92.9%  (10 items)
  speaker 5: CER  56.1%  WER  94.8%  (10 items)
best item (CER  31.2%):
  REF Сан-Франциско мэрі Лондон Брид Трампты саясаттан басқа себепсіз өз қаласын сынауға айналдырды
  HYP Сан Франсиска Мері Лондон-Брит Трамты сая саттам басқа севепсіз уезаласынсы
median item (CER  57.4%):
  REF Оны қорғаушы «ойынды мерзімінен бұрын аяқтауға тырыспады» деп айыптады.
  HYP Оны қорауша оындым ежеменен брын ая
worst item (CER  94.6%):
  REF Дж. Ло, Shakira’s Super Bowl брейк-шоуындағы өнерлері үшін.
  HYP лііііііі
```

**The baseline is very weak.** On 7 minutes of clean synthetic Kazakh speech, Whisper small with the `kk` token has a normalised CER of 56.4% and a WER of 93.4%: total word edit operations equal 93.4% of the reference word count. This is not the percentage of reference words containing errors, because insertions also contribute. Normalisation (lower case, punctuation removed, apostrophes and letters kept) lowers WER only from 96.2% to 93.4%, so the errors are not a matter of casing or punctuation. The examples show the typical pattern: words are recognised by sound but misspelt (*Франциско* → *Франсиска*, *себепсіз* → *севепсіз*), word boundaries move (*саясаттан* → *сая саттам*), and 19 of the 60 hypotheses are less than half as long as their reference: the transcripts are substantially shorter than their references. This character-count check alone does not establish when or why decoding ended. The worst item contains English words (*Shakira's Super Bowl*) and decodes to a run of repeated letters, a typical failure of autoregressive decoding.

**Per-speaker results** range from 51.2% to 61.6% CER over ten items each. With ten items per voice that difference is not reliable on its own (Exercise 3 repeats the comparison with the same sentences for every voice).

**What a real comparison needs.** This test set is synthetic, so it says nothing about real Kazakh recordings; it does show the protocol. With `code/remaining/speech.py` the baseline and adapted checkpoints are decoded on the same complete FLEURS test items, with the language token, task and generation limit fixed, and the report contains the same raw and normalised WER and CER, the retained duration and per-item transcripts. Report the number of items, the audio duration and the decoding settings with every score.

## Your turn — extension

1. Decode the same items with `small-encoder.onnx`/`small-decoder.onnx` (the float32 export, about twice as slow) and measure what the int8 quantisation costs.
2. Write a normalisation that also removes the hyphen in compounds such as *Сан-Франциско*. How much does WER change, and should a benchmark report it?
3. Record ten sentences yourself (with consent), transcribe them and compare the error pattern with the synthetic one.